## 03 · Trigger initial sync and wait READY

Pre-requisites (must be complete before running this notebook):
- `01_symptom_corpus.py` — `workspace.default.symptom_corpus` exists, CDF=true
- `02_vs_endpoint.py` — VS endpoint online, Delta-sync index created (TRIGGERED)

**No agent code may run until `INDEX_READY = True` at the bottom of this notebook.**

In [ ]:
import json
import os
import subprocess
import sys
import time

PROFILE    = os.environ.get("DBX_PROFILE", "tero2")
INDEX_NAME = "workspace.default.symptom_corpus_idx"

READY_STATES   = {"ONLINE", "ONLINE_NO_PENDING_UPDATE"}
POLL_INTERVAL  = 15   # seconds between status checks
POLL_MAX       = 80   # max iterations ≈ 20 min

print(f"Index : {INDEX_NAME}")
print(f"Profile: {PROFILE}")

In [ ]:
def api(method: str, path: str, body=None) -> dict:
    """Thin wrapper around `databricks api`; tolerates non-zero exit when stdout
    is valid JSON (VS API returns 4xx on benign cases like already-syncing)."""
    cmd = ["databricks", "api", method, "-p", PROFILE, path]
    if body is not None:
        cmd += ["--json", json.dumps(body)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = r.stdout or ""
    if r.returncode != 0 and not out.strip():
        raise RuntimeError(
            f"api({method} {path}) transport error rc={r.returncode}: "
            f"{(r.stderr or '').strip()[:300]}"
        )
    if not out.strip():
        return {}
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        raise RuntimeError(f"api({method} {path}) non-JSON response: {out[:300]}")

In [ ]:
# ── Trigger initial sync ──────────────────────────────────────────────────────
# TRIGGERED pipeline type does NOT auto-sync on table write; an explicit POST
# /sync is required.  We tolerate HTTP 4xx (already syncing) gracefully.

print(f"→ triggering sync on {INDEX_NAME!r}...")
sync_resp = api("post", f"/api/2.0/vector-search/indexes/{INDEX_NAME}/sync")
print(f"  sync response: {sync_resp or '(empty — likely already syncing, that is OK)'}")

In [ ]:
# ── Poll until READY ──────────────────────────────────────────────────────────
# READY = detailed_state in {ONLINE, ONLINE_NO_PENDING_UPDATE}
# No agent code runs until INDEX_READY is True.

INDEX_READY = False

print(f"→ polling index state (up to {POLL_MAX * POLL_INTERVAL // 60} min)...")
for i in range(POLL_MAX):
    idx = api("get", f"/api/2.0/vector-search/indexes/{INDEX_NAME}")
    status        = idx.get("status", {})
    detail_state  = status.get("detailed_state", "?")
    indexed_rows  = status.get("indexed_row_count", 0)
    message       = status.get("message", "")

    print(f"  [{i * POLL_INTERVAL:>4}s] state={detail_state:<35} indexed={indexed_rows}")

    if detail_state in READY_STATES:
        INDEX_READY = True
        break

    if "FAILED" in detail_state:
        raise RuntimeError(
            f"Index entered failure state {detail_state!r}: {message or idx}"
        )

    time.sleep(POLL_INTERVAL)

if not INDEX_READY:
    raise RuntimeError(
        f"Index {INDEX_NAME!r} did not reach READY within "
        f"{POLL_MAX * POLL_INTERVAL // 60} min.  "
        "Last state: " + detail_state
    )

In [ ]:
# ── READY gate ────────────────────────────────────────────────────────────────
# This assertion is the hard boundary — downstream agent code MUST NOT be
# placed above this cell.  If INDEX_READY is False (timeout or failure),
# the RuntimeError above already aborted the notebook.

assert INDEX_READY, (
    f"INDEX_READY={INDEX_READY!r} — "
    "do not proceed with agent code until the index is READY."
)

print(f"\n✓ {INDEX_NAME} is READY")
print(f"  state         = {detail_state}")
print(f"  indexed_rows  = {indexed_rows}")
print("\n  Agent code is now safe to run.")